> ## SOLUTION / ANSWER KEY &mdash; Lab 7.2
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-02-gather-context.ipynb`](../lab-02-gather-context.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.2 &mdash; Gather Context with Tools

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Wrap lookup_order and get_template as @tool functions
- Gather the order + the reply template BEFORE drafting
- Watch a real Groq agent call YOUR gather tools from the trace

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Grounding the task in real data](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = "/tmp/biaa-lab-07-02"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
A general model doesn't know **your** client's order or **your** reply templates &mdash; so the agent
must **gather context first, then draft** (deck slide 6). Gathering happens through **tools** over
your systems: an orders DB, a template store, the CRM. An agent that drafts before it gathers
**hallucinates specifics**; one that grounds every claim in retrieved context is accurate and
auditable. Here you build real `@tool`s and watch a **real agent** call them.

In [ ]:
from langchain_core.tools import tool

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

print("orders on file :", list(ORDERS))
print("templates on file:", list(TEMPLATES))

## Build it
Complete the two gather tools and the `gather` step that pulls both **before** any drafting.

In [ ]:
from langchain_core.tools import tool

@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order's status, ETA and carrier by id."""
    return ORDERS.get(order_id, {"status": "unknown"})

@tool
def get_template(kind: str) -> str:
    """Fetch a reply template by kind, e.g. delivery_delay or refund."""
    return TEMPLATES.get(kind, "")

def gather(order_id, kind):
    # gather FIRST: pull the order AND the template before we draft anything
    return {"order": lookup_order.invoke(order_id), "template": get_template.invoke(kind)}

try:
    print("order 4471 :", lookup_order.invoke("4471"))
    print("unknown    :", lookup_order.invoke("9999"))
    print("gathered   :", gather("4471", "delivery_delay"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Bind your two gather tools to a real Groq agent and ask it to gather order 4471. Read the trace: the model calls YOUR functions to ground itself before answering.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to the repo .env (free key at console.groq.com), then re-run this cell.")
else:
    try:
        from langchain.agents import create_agent
        agent = create_agent(llm, tools=[lookup_order, get_template])
        result = with_backoff(lambda: agent.invoke(
            {"messages": [("user", "Gather the status and ETA of order 4471 for a delivery update. Do not send anything.")]},
            config={"recursion_limit": 8}))
        print_trace(result)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The trace shows **`TOOL CALL: lookup_order {'order_id': '4471'}`** &mdash; the real model chose to call your Python.
- Each **`OBS:`** line is what your tool returned; the model reads it, then answers with grounded specifics.
- Gather-first is why the reply can say "due Friday" truthfully &mdash; it retrieved that, it didn't invent it.

## Your turn (open task &mdash; no grader)
Ask the agent about an order that isn't on file (e.g. `9999`), or add a third gather tool (say
`order_history(order_id)`), bind it, and re-run. **What good looks like:** for the unknown order the tool
returns `{"status": "unknown"}` and the agent says so honestly instead of inventing an ETA.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
if groq_ready():
    from langchain.agents import create_agent
    agent = create_agent(llm, tools=[lookup_order, get_template])
    result = with_backoff(lambda: agent.invoke(
        {"messages": [("user", "What is the status of order 9999? Do not send anything.")]},
        config={"recursion_limit": 8}))
    print_trace(result)   # lookup_order returns {"status": "unknown"}; the agent says so honestly
else:
    print("(add GROQ_API_KEY to .env)")

---
### Key takeaway
Gather first, draft second. These are just tools pointed at a real job -- and a real Groq agent calls them to ground itself before it writes a word.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>